In [ ]:
import MDAnalysis as mda

try:
    u = mda.Universe("/data/cg_dna_histone/combined/dna_initial.pdb", "/data/cg_dna_histone/combined/dna_traj.dcd")
    for ts in u.trajectory:
        pass  # Attempts to read every single frame to the end
    print("File is intact!")
except Exception as e:
    print(f"Corruption detected: {e}")


In [ ]:
import os
import struct
import numpy as np
import MDAnalysis as mda

# Paths - Update these to match your actual files
topology_path = "/data/cg_dna_histone/combined/dna_initial.pdb"
corrupt_dcd_path = "/data/cg_dna_histone/combined/dna_traj.dcd"
fixed_dcd_path = "/data/cg_dna_histone/combined/dna_traj_fixed.dcd"

# Load the topology to get a clean universe structure
print("Loading topology...")
u = mda.Universe(topology_path)
n_atoms = len(u.atoms)  # This will be 27464

# Calculate expected block sizes in the binary file
# Each frame contains X, Y, and Z blocks. Each block has a 4-byte header and 4-byte footer.
coord_block_size = n_atoms * 4  # 4 bytes per float32 coordinate
frame_data_size = 3 * (4 + coord_block_size + 4) 

# Check if your simulation used Periodic Boundary Conditions (Unit Cell box)
# If yes, each frame has an additional 56-byte unit cell block at the front.
# We will check both possibilities dynamically.
file_size = os.path.getsize(corrupt_dcd_path)

print(f"File size: {file_size / (1024**3):.2f} GB")
print(f"Atoms: {n_atoms}")

def recover_dcd(has_box=False):
    actual_frame_size = frame_data_size + (56 if has_box else 0)
    
    # Estimate total frames based on file size (skipping roughly 276 bytes for standard header)
    estimated_frames = int((file_size - 276) / actual_frame_size)
    print(f"Attempting recovery assuming PBC Box={has_box}. Estimated frames: {estimated_frames}")
    
    with open(corrupt_dcd_path, "rb") as f_in:
        # Read the whole file into memory as bytes (or memory-map it if RAM is low)
        # To be safe with 11GB, we scan sequentially:
        
        # 1. Let's find where the actual coordinates begin by scanning for the first block size signature
        # A standard block size integer for 27464 atoms is 27464 * 4 = 109856
        target_signature = struct.pack("I", coord_block_size)
        content = f_in.read(10000) # Read the first 10KB to find the header end
        
        start_idx = content.find(target_signature)
        if start_idx == -1:
            return False
        
        # Rewind and align to the start of the first frame
        # The signature is preceded by 4 bytes (the block size itself) or box coordinates
        first_frame_start = start_idx - 4 if not has_box else start_idx - 52
        
        f_in.seek(first_frame_start)
        
        # 2. Write out clean frames using MDAnalysis Writer
        with mda.Writer(fixed_dcd_path, n_atoms) as w:
            frames_written = 0
            while True:
                try:
                    # If it has a box, skip or read the 56 box bytes
                    if has_box:
                        box_data = f_in.read(56)
                        if len(box_data) < 56: break
                    
                    # Read X coordinates
                    hx = f_in.read(4)
                    if len(hx) < 4: break
                    x_bytes = f_in.read(coord_block_size)
                    fx = f_in.read(4)
                    
                    # Read Y coordinates
                    hy = f_in.read(4)
                    y_bytes = f_in.read(coord_block_size)
                    fy = f_in.read(4)
                    
                    # Read Z coordinates
                    hz = f_in.read(4)
                    z_bytes = f_in.read(coord_block_size)
                    fz = f_in.read(4)
                    
                    if len(z_bytes) < coord_block_size: break
                    
                    # Convert raw bytes to float32 arrays
                    x = np.frombuffer(x_bytes, dtype=np.float32)
                    y = np.frombuffer(y_bytes, dtype=np.float32)
                    z = np.frombuffer(z_bytes, dtype=np.float32)
                    
                    # Combine into standard MDAnalysis coordinate format (N, 3)
                    coords = np.vstack([x, y, z]).T
                    
                    # Update universe coordinates and write frame
                    u.atoms.positions = coords
                    w.write(u)
                    
                    frames_written += 1
                    if frames_written % 1000 == 0:
                        print(f"Recovered {frames_written} frames...")
                        
                except Exception as e:
                    print(f"Stopped writing at frame {frames_written}: {e}")
                    break
                    
            print(f"Success! Finished recovering {frames_written} frames into {fixed_dcd_path}")
            return True

# Run recovery try 1 (Without Box)
success = recover_dcd(has_box=False)
# If that extracts 0 frames, it means your simulation used a box, run try 2:
if not success:
    print("Regrouping... trying with Box dimensions enabled.")
    recover_dcd(has_box=True)


In [ ]:
import struct

corrupt_dcd_path = "/data/cg_dna_histone/combined/dna_traj.dcd"

with open(corrupt_dcd_path, "rb") as f:
    header_bytes = f.read(100)

print("--- First 100 raw bytes as integers (4-byte chunks) ---")
for i in range(0, 100, 4):
    chunk = header_bytes[i:i+4]
    if len(chunk) == 4:
        val_int = struct.unpack("I", chunk)[0]
        val_float = struct.unpack("f", chunk)[0]
        print(f"Byte {i:02d}: Int = {val_int:<12} | Float = {val_float:.4e}")


In [ ]:
import MDAnalysis as mda
from MDAnalysis.coordinates.DCD import DCDReader

topology_path = "/data/cg_dna_histone/combined/dna_initial.pdb"
corrupt_dcd_path = "/data/cg_dna_histone/combined/dna_traj.dcd"
fixed_dcd_path = "/data/cg_dna_histone/combined/dna_traj_fixed.dcd"

# Load base universe structure
u = mda.Universe(topology_path)
n_atoms = len(u.atoms)

print("Opening direct trajectory stream...")
frames_written = 0

# Open low-level DCDReader directly to bypass Universe integrity checks
with DCDReader(corrupt_dcd_path) as r:
    with mda.Writer(fixed_dcd_path, n_atoms) as w:
        # Loop over every single frame that physically exists in the 11 GB file
        for frame in r:
            try:
                # Update positions in our universe container
                u.atoms.positions = frame.positions
                
                # If your simulation saved box dimensions, copy them over
                if frame.has_dimensions:
                    u.trajectory.ts.dimensions = frame.dimensions
                
                # Write to the new, healthy DCD file
                w.write(u)
                frames_written += 1
                
                if frames_written % 1000 == 0:
                    print(f"Successfully recovered {frames_written} frames...")
            except Exception as frame_error:
                print(f"Reached broken file boundary at frame {frames_written}: {frame_error}")
                break

print(f"\nFinished! Successfully salvaged {frames_written} frames into {fixed_dcd_path}")


In [ ]:
import os
import struct
import numpy as np
import MDAnalysis as mda

topology_path = "/data/cg_dna_histone/combined/dna_initial.pdb"
corrupt_dcd_path = "/data/cg_dna_histone/combined/dna_traj.dcd"
fixed_dcd_path = "/data/cg_dna_histone/combined/dna_traj_fixed.dcd"

# Load the base universe layout
u = mda.Universe(topology_path)
n_atoms = len(u.atoms)  # 27464

# Binary size definitions
coord_block_bytes = n_atoms * 4  # 109856 bytes per coordinate component (X, Y, Z)

print("Scanning binary structure...")

with open(corrupt_dcd_path, "rb") as f_in:
    # 1. Skip past the main header block (84 bytes body + 8 bytes boundary tags = 92 bytes)
    f_in.seek(92)
    
    # 2. Read the title/remarks block dimension marker
    title_size_bytes = f_in.read(4)
    if not title_size_bytes:
        raise ValueError("Could not read title section marker")
    
    title_size = struct.unpack("I", title_size_bytes)[0]
    print(f"Title section body size: {title_size} bytes")
    
    # Skip the title body plus its trailing 4-byte boundary tag
    f_in.seek(title_size + 4, 1)
    
    # 3. Read the atom count verification block (4 bytes header + 4 bytes integer + 4 bytes footer = 12 bytes)
    atom_check_block = f_in.read(12)
    verified_atoms = struct.unpack("I", atom_check_block[4:8])[0]
    print(f"Header verifies frame atom count: {verified_atoms}")
    
    # Current file pointer position is exactly where the first frame starts!
    first_frame_offset = f_in.tell()
    print(f"Coordinates begin at byte index: {first_frame_offset}")

    # 4. Probe the first frame to check if Periodic Boundary Conditions (Box Dimensions) are saved
    # If a box is present, the frame starts with a 48-byte box block surrounded by two 4-byte tags (56 bytes total)
    # The tag value will be exactly 48.
    box_tag_bytes = f_in.read(4)
    has_box = False
    if len(box_tag_bytes) == 4:
        box_tag = struct.unpack("I", box_tag_bytes)[0]
        if box_tag == 48:
            has_box = True
            
    print(f"Simulation details -> Includes Box Dimensions (PBC): {has_box}")
    
    # Rewind back to the start of the frames
    f_in.seek(first_frame_offset)
    
    # 5. Extract and rewrite into a healthy DCD file
    frames_written = 0
    with mda.Writer(fixed_dcd_path, n_atoms) as w:
        while True:
            try:
                # If the trajectory has box metrics, read or skip the 56-byte unit cell frame header
                if has_box:
                    box_data = f_in.read(56)
                    if len(box_data) < 56: break
                    # Parse box dimensions matrix to keep the information intact
                    box_floats = struct.unpack("d"*6, box_data[4:52])
                    u.trajectory.ts.dimensions = [*box_floats[0:3], *box_floats[3:6]]
                
                # --- Read Coordinate Components ---
                # X coordinates
                hx = f_in.read(4)
                if len(hx) < 4: break
                x_bytes = f_in.read(coord_block_bytes)
                fx = f_in.read(4)
                
                # Y coordinates
                hy = f_in.read(4)
                y_bytes = f_in.read(coord_block_bytes)
                fy = f_in.read(4)
                
                # Z coordinates
                hz = f_in.read(4)
                z_bytes = f_in.read(coord_block_bytes)
                fz = f_in.read(4)
                
                if len(z_bytes) < coord_block_bytes:
                    break  # Hit EOF or partial frame write
                
                # Convert raw memory buffers to float32 numpy arrays
                x = np.frombuffer(x_bytes, dtype=np.float32)
                y = np.frombuffer(y_bytes, dtype=np.float32)
                z = np.frombuffer(z_bytes, dtype=np.float32)
                
                # Reconstruct into standard (N, 3) matrix format
                u.atoms.positions = np.vstack([x, y, z]).T
                
                # Write to the fresh, uncorrupted output stream
                w.write(u)
                frames_written += 1
                
                if frames_written % 1000 == 0:
                    print(f"Successfully carved {frames_written} frames...")
                    
            except Exception as e:
                print(f"Data stream boundary reached at frame {frames_written}: {e}")
                break

print(f"\nSuccess! Recovered {frames_written} frames cleanly into: {fixed_dcd_path}")


In [3]:
import MDAnalysis as mda, numpy as np

top = "/data/cg_dna_histone/combined/Mg8/4_day_run_1/nucleosome_sim_1/run1/dna_initial.pdb"
dcd = "/data/cg_dna_histone/combined/Mg8/4_day_run_1/nucleosome_sim_1/run1/dna_traj.dcd"

u = mda.Universe(top, dcd)
n = len(u.trajectory)
print("frames:", n, "| atoms:", u.atoms.n_atoms)
print("box (frame 0):", u.trajectory[0].dimensions)
print("dt:", u.trajectory.dt)

# first 5 frames + 20 frames evenly spread over the whole run
idx = sorted(set(range(5)) | set(np.linspace(0, n - 1, 20, dtype=int).tolist()))
with mda.Writer("slice.dcd", u.atoms.n_atoms) as w:
    for i in idx:
        u.trajectory[i]
        w.write(u.atoms)
print("wrote", len(idx), "frames:", idx)

/home/feynman/miniconda3/envs/venv/lib/python3.13/site-packages/MDAnalysis/coordinates/DCD.py:171: DeprecationWarning: DCDReader currently makes independent timesteps by copying self.ts while other readers update self.ts inplace. This behavior will be changed in 3.0 to be the same as other readers. Read more at https://github.com/MDAnalysis/mdanalysis/issues/3889 to learn if this change in behavior might affect you.
  warnings.warn("DCDReader currently makes independent timesteps"
/home/feynman/miniconda3/envs/venv/lib/python3.13/site-packages/MDAnalysis/coordinates/DCD.py:493: UserWarning: No dimensions set for current frame, zeroed unitcell will be written
  warnings.warn(wmsg)


frames: 22068 | atoms: 27464
box (frame 0): None
dt: 10.000000029814057
wrote 24 frames: [0, 1, 2, 3, 4, 1161, 2322, 3484, 4645, 5807, 6968, 8129, 9291, 10452, 11614, 12775, 13937, 15098, 16259, 17421, 18582, 19744, 20905, 22067]
